# IOAI — 2025 Selection Camp Hotspot (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/Satellite_Images-1'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/roai-hotspot.zip', '/tmp/hs.zip')
    with zipfile.ZipFile('/tmp/hs.zip') as zf: zf.extractall('data')
print('데이터 준비:', 'data/Satellite_Images-1' if os.path.exists('data/Satellite_Images-1') else '실패')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# HotSpot — 위성영상 이진 분할 (베이스라인)

> 데이터는 DGX/Colab에 **자동 준비**됩니다 — `data/Satellite_Images-1~4`(각 251장 256×256). 검은 배경 위 색 도형을 **이진 분할**해 RLE 로 제출합니다(`submission.csv`: subtaskID, datapointID, answer).

베이스라인: **그레이스케일 + Otsu 전역 임계값**(노이즈/난이도별 튜닝 없음). 난이도가 올라가는 Satellite_Images-3·4 에서 노이즈에 약합니다. (개선: 노이즈별 median blur + 적응 임계값 → **모범답안**, OpenCV만으로 100점.)

> ⚠️ 자동 채점은 미제공(원대회 test 정답 마스크 비공개) — 풀이를 **원대회 judge**에 제출해 IoU 채점하세요. 여기선 코드 작성·실행 연습용.

In [ ]:
import os, numpy as np, pandas as pd, cv2
from PIL import Image
from tqdm import tqdm
root_path = 'data'

In [ ]:
def image_to_rle(img_arr: np.array) -> list[int]:
    side, side = img_arr.shape[0], img_arr.shape[1]
    img_arr = img_arr.reshape(side*side, order='F')
    
    start = 0
    length = 0
    res = []
    for i in range(side * side):
        if img_arr[i] == 255:
            if length == 0:
                start = i + 1
            length = length + 1
        else:
            if length > 0:
                res.append(start)
                res.append(length)
            length = 0
    if length > 0:
        res.append(start)
        res.append(length)
        
    return res

In [ ]:
def simple_mask(img):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    _, mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return mask
rows = []
for st in range(1, 5):
    path = f'{root_path}/Satellite_Images-{st}'
    for fn in tqdm(sorted(os.listdir(path)), desc=f'subtask {st}'):
        img = np.array(Image.open(f'{path}/{fn}').convert('RGB'))
        rows.append({'subtaskID': st, 'datapointID': fn, 'answer': image_to_rle(simple_mask(img))})
pd.DataFrame(rows).to_csv('submission.csv', index=False)
print('wrote submission.csv', len(rows))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv', 'submission.zip', 'submission.jsonl', 'submission.json']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)